# 02 Feature Engineering

In [25]:
import pandas as pd
import numpy as np

df_clean = pd.read_parquet("../../data/df_clean_raw.parquet")

## Maßnahmenraum (Hypothesen)

Der Smart Service soll der Verkehrsbehörde nicht nur zeigen, wo schwere Unfälle passieren, sondern welche Maßnahme dort sinnvoll ist. Deshalb wird vor der Datenanalyse festgelegt, welche Maßnahmen grundsätzlich in Frage kommen. Grundlage sind etablierte Maßnahmenkataloge der Verkehrssicherheit (WHO *Save LIVES*, iRAP *Road Safety Toolkit*), gegliedert nach dem 3E-Modell (Engineering, Enforcement, Education).

Jede Maßnahme setzt eine bestimmte Problemsituation voraus. Daraus ergibt sich eine Hypothese, die mit den Daten geprüft wird, und das Merkmal (Feature Engineering), das dafür benötigt wird.

| Nr. | Hypothese: Unfälle sind häufiger schwer, wenn … | Mögliche Maßnahme | 3E | Merkmal |
|---|---|---|---|---|
| H1 | … sie abends oder nachts passieren | Abendliche Geschwindigkeitskontrollen | Enforcement | `Abend_Nacht`, `Tageszeit` |
| H2 | … sie auf freier Strecke (außerhalb von Knotenpunkten) passieren | Streckenbezogene Geschwindigkeitsüberwachung, Leitpfosten | Enforcement, Engineering | `Freie_Strecke` |
| H3 | … sie in Wohn- oder Bürogebieten passieren | Tempo 30, Verkehrsberuhigung | Engineering | `Area_accident_occured` |
| H4 | … es dunkel ist und keine Beleuchtung vorhanden ist | Straßenbeleuchtung nachrüsten | Engineering | `Light_conditions` |
| H5 | … Fußgänger beteiligt sind, besonders bei Dunkelheit | Beleuchtete Fußgängerüberwege, Mittelinseln | Engineering | `Kollision`, `Light_conditions` |
| H6 | … ein Fahrzeug mit einem Objekt am Straßenrand kollidiert | Schutzplanken, Absicherung von Hindernissen | Engineering | `Kollision` |
| H7 | … der Fahrer jünger als 18 Jahre ist | Führerscheinkontrollen, Halterhaftung | Enforcement | `Age_band_of_driver` |
| H8 | … die Ursache Überholen oder Fahren auf der Gegenspur ist | Überholverbote, Mitteltrennung | Engineering, Enforcement | `Cause_of_accident` |
| H9 | … der Fahrer wenig Fahrerfahrung hat | Schulungen für Fahranfänger | Education | `Driving_experience` |
| H10 | … es regnet oder die Fahrbahn nass ist | Griffige Fahrbahnbeläge, Warnhinweise | Engineering, Education | `Weather_conditions`, `Road_surface_conditions` |
| H11 | … sie am Wochenende passieren | Verstärkte Kontrollen am Wochenende | Enforcement | `Wochenende` |

**Prüfkriterium:** Eine Hypothese gilt als bestätigt, wenn der Anteil schwerer Unfälle in der betreffenden Gruppe deutlich über dem Durchschnitt liegt (95-%-Konfidenzintervall oberhalb des Durchschnitts, mindestens 50 Unfälle) und sich daraus ein relevanter Überschuss schwerer Unfälle ergibt.

**Quellen der Hypothesen:**
- World Health Organization (2017): *Save LIVES – A Road Safety Technical Package*. Genf: WHO. https://www.who.int/en/publications/i/item/save-lives-a-road-safety-technical-package
- International Road Assessment Programme (iRAP): *Road Safety Toolkit*. https://toolkit.irap.org (abgerufen am 12.11.2026)

## Feature Engineering
| Neues Merkmal | Bedeutung | Zweck |
|---|---|---|
| `schwer` | 1 = schwer oder tödlich, 0 = leicht | Zielvariable |
| `Stunde`, `Tageszeit` | Uhrzeit in 5 Zeitfenster gruppiert | Zeitliche Planung von Maßnahmen |
| `Wochenende` | Samstag/Sonntag | Prüfung der Hypothese „Wochenende" |
| `Abend_Nacht` | Unfall zwischen 20 und 5 Uhr | Zeitfenster für Kontrollen und Beleuchtung |
| `Freie_Strecke` | Unfall außerhalb eines Knotenpunkts | Strecken- statt Kreuzungsmaßnahmen |
| `Fahrzeuggruppe` | 17 Fahrzeugtypen in 5 Gruppen | Fahrzeugspezifische Maßnahmen |

In [26]:
# Zielvariable
df_clean["schwer"] = (df_clean["Accident_severity"] != "Slight Injury").astype(int)
schwere_rang = {"Slight Injury": 0, "Serious Injury": 1, "Fatal injury": 2}
df_clean["Schwere_Rang"] = df_clean["Accident_severity"].map(schwere_rang)

# Zeit
df_clean["Stunde"] = pd.to_datetime(df_clean["Time"], format="%H:%M:%S", errors="coerce").dt.hour
df_clean["Tageszeit"] = pd.cut(df_clean["Stunde"], bins=[-1, 5, 9, 15, 19, 23],
    labels=["Nacht (0-5)", "Morgen (6-9)", "Tag (10-15)", "Abend-Rush (16-19)", "Abend (20-23)"])
df_clean["Wochenende"] = df_clean["Day_of_week"].isin(["Saturday", "Sunday"]).astype(int)
df_clean["Abend_Nacht"] = ((df_clean["Stunde"] >= 20) | (df_clean["Stunde"] <= 5)).astype(int)

# Ort
df_clean["Freie_Strecke"] = (df_clean["Types_of_Junction"] == "No junction").astype(int)

# Fahrzeuggruppe
def fahrzeuggruppe(t):
    if t in ["Automobile", "Stationwagen", "Taxi"]: return "Pkw"
    if "Lorry" in t or "lorry" in t or "Pick up" in t: return "Lkw/Pick-up"
    if "Public" in t: return "Bus"
    if t in ["Motorcycle", "Bicycle", "Bajaj"]: return "Zweirad/Dreirad"
    return "Sonstige/Unknown"
df_clean["Fahrzeuggruppe"] = df_clean["Type_of_vehicle"].apply(fahrzeuggruppe)

### Unfall-ID
Jede Zeile im Datensatz beschreibt ein beteiligtes Fahrzeug, nicht einen Unfall. Ein Unfall mit mehreren Fahrzeugen erscheint daher mehrfach. Die Unfall-ID fasst diese Zeilen zusammen, damit jeder Unfall nur einmal gezählt wird.

Da der Datensatz keine Unfallnummer enthält, wird die ID aus sieben Merkmalen gebildet, die für alle Fahrzeuge eines Unfalls identisch sind (Uhrzeit, Wochentag, Anzahl Fahrzeuge, Anzahl Verunglückte, Gebiet, Wetter, Licht). Haben Zeilen eines Unfalls unterschiedliche Schwere, zählt die höchste.

In [27]:
key = ["Time", "Day_of_week", "Number_of_vehicles_involved", "Number_of_casualties",
       "Area_accident_occured", "Weather_conditions", "Light_conditions"]
df_clean["Unfall_ID"] = df_clean[key].astype(str).agg("|".join, axis=1).factorize()[0]

# Ein Unfall = eine Zeile; bei unterschiedlicher Schwere zählt die höchste
df_unfall = (df_clean.sort_values("Schwere_Rang", ascending=False)
             .groupby("Unfall_ID").first().reset_index())
print("Zeilen (Fahrzeuge):", len(df_clean))
print("Unfälle:", len(df_unfall))

Zeilen (Fahrzeuge): 12316
Unfälle: 9517


## Speichern

In [28]:
df_clean.to_parquet("../../data/df_features.parquet", index=False)
df_unfall.to_parquet("../../data/df_unfall.parquet", index=False)
print("Gespeichert: df_features.parquet", df_clean.shape)
print("Gespeichert: df_unfall.parquet", df_unfall.shape)

Gespeichert: df_features.parquet (12316, 41)
Gespeichert: df_unfall.parquet (9517, 41)
